<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<span style="display:inline-block;margin-bottom:10px;padding:2px 12px;border-radius:999px;background:#6366f1;color:#fff;font:600 11px/20px system-ui,sans-serif;letter-spacing:.05em;text-transform:uppercase">Notebook 05</span>
<h1 style="font-size:1.35em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">Deterministic tool calling (live voice + agentic)</h1>
<p style="margin:.55em 0">The model <b>never runs code</b>. It can only <i>propose</i> a call: a tool name and JSON arguments.
Everything after that is ordinary Python that <b>we</b> control:</p>
<div style="display:flex;flex-direction:column;align-items:flex-start;gap:4px;margin:.8em 0">
  <div style="border:1px solid rgba(127,127,127,.35);background:rgba(127,127,127,.05);border-radius:8px;padding:8px 12px"><b>you speak</b> → model sends <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">response.done</code> with <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">output: [function_call(name, call_id, arguments)]</code></div>
  <div style="opacity:.6;padding-left:18px">▼</div>
  <div style="border:2px solid #6366f1;background:rgba(127,127,127,.05);border-radius:8px;padding:8px 12px"><b>our code:</b> allowlist name → bound size → parse JSON → validate schema → run fn<br>
    <span style="opacity:.7;font-size:.9em">any failure becomes <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">{"error": ...}</code>, never a crash</span></div>
  <div style="opacity:.6;padding-left:18px">▼</div>
  <div style="border:1px solid rgba(127,127,127,.35);background:rgba(127,127,127,.05);border-radius:8px;padding:8px 12px"><code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">conversation.item.create</code> with <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">function_call_output(call_id, output)</code>, then <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">response.create</code></div>
  <div style="opacity:.6;padding-left:18px">▼</div>
  <div style="border:1px solid rgba(127,127,127,.35);background:rgba(127,127,127,.05);border-radius:8px;padding:8px 12px"><b>model speaks the result</b></div>
</div>
<div style="margin:.8em 0;padding:.4em 1em;border-left:4px solid #6366f1;background:rgba(99,102,241,.12);border-radius:0 6px 6px 0">The model decides <i>what</i> to ask for. Our code decides <i>what happens</i>: that is the deterministic part.</div>
<p style="margin:.55em 0">New helpers: <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">live/tools.py</code> (spec, validate, run), <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">realtime.answer_tool_calls</code>, and the
<code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">tools=</code> parameter on <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">session_config</code> and <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">converse</code>. RAG would be just one more tool in this registry.</p>

</div>

In [1]:
import json
from collections import Counter
from IPython.display import Audio
from live import audio, realtime, tools

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<h2 style="font-size:1.2em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">1. Tools are plain Python functions</h2>
<p style="margin:.55em 0">Normalization happens in <b>our</b> code, at the boundary. Speech gives you "a-123", "A 123" and so on, and the tool accepts them all.</p>

</div>

In [2]:
ORDERS = {"A123": {"status": "shipped", "arrives": "Thursday"},
          "B456": {"status": "processing", "arrives": "next week"}}

def get_order_status(order_id):
    key = "".join(ch for ch in order_id.upper() if ch.isalnum())
    if key not in ORDERS:
        raise LookupError(f"no order {key}")
    return ORDERS[key]

def convert_temperature(value, to):
    return round(value * 9 / 5 + 32, 1) if to == "F" else round((value - 32) * 5 / 9, 1)

get_order_status("a-123"), convert_temperature(30, "F")

({'status': 'shipped', 'arrives': 'Thursday'}, 86.0)

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<h2 style="font-size:1.2em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">2. The registry: the allowlist of what the model may ask for</h2>
<p style="margin:.55em 0">Each entry pairs a JSON schema (sent to the model) with the function (kept on our side).</p>

</div>

In [3]:
REGISTRY = {
    "get_order_status": (tools.spec("get_order_status", "Look up a customer's order by its ID, e.g. A123.",
                                    {"order_id": {"type": "string", "description": "Order ID like A123"}}),
                         get_order_status),
    "convert_temperature": (tools.spec("convert_temperature", "Convert a temperature to Celsius (C) or Fahrenheit (F).",
                                       {"value": {"type": "number"}, "to": {"type": "string", "enum": ["C", "F"]}}),
                            convert_temperature),
}
SPECS = [s for s, _ in REGISTRY.values()]
SPECS[1]

{'type': 'function',
 'name': 'convert_temperature',
 'description': 'Convert a temperature to Celsius (C) or Fahrenheit (F).',
 'parameters': {'type': 'object',
  'properties': {'value': {'type': 'number'},
   'to': {'type': 'string', 'enum': ['C', 'F']}},
  'required': ['value', 'to'],
  'additionalProperties': False}}

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<h2 style="font-size:1.2em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">3. Our side of the contract: every bad proposal becomes data, not a crash</h2>
<p style="margin:.55em 0">The model's arguments are untrusted input. Feed <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">tools.run</code> the kinds of things a model might propose.</p>
<div style="margin:.8em 0;padding:.4em 1em;border-left:4px solid #6366f1;background:rgba(99,102,241,.12);border-radius:0 6px 6px 0">The red traceback below is <b>our</b> log. A caught error must leave evidence for the developer,
while the model only receives the short <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">error</code> string.</div>

</div>

In [4]:
proposals = [
    ("get_order_status",    '{"order_id": "A123"}'),               # good
    ("get_order_status",    '{"order_id": "Z999"}'),               # valid, but no such order
    ("convert_temperature", '{"value": 30, "to": "K"}'),           # not in enum
    ("convert_temperature", '{"value": "30", "to": "F"}'),         # wrong type
    ("get_order_status",    '{"order_id": "A1", "admin": true}'),  # extra argument
    ("delete_all_orders",   '{}'),                                 # not in the registry
    ("get_order_status",    '{"order_id": '),                      # truncated JSON
]
[(name, tools.run(REGISTRY, name, args)) for name, args in proposals]

ERROR:root:tool get_order_status failed
Traceback (most recent call last):
  File "D:\Code\realtime-talk\live\tools.py", line 54, in run
    return {"result": fn(**args)}
                      ^^^^^^^^^^
  File "C:\Users\aksha\AppData\Local\Temp\ipykernel_26548\2578210558.py", line 7, in get_order_status
    raise LookupError(f"no order {key}")
LookupError: no order Z999


[('get_order_status',
  {'result': {'status': 'shipped', 'arrives': 'Thursday'}}),
 ('get_order_status', {'error': 'LookupError: no order Z999'}),
 ('convert_temperature', {'error': "to: must be one of ['C', 'F']"}),
 ('convert_temperature', {'error': 'value: expected number, got str'}),
 ('get_order_status', {'error': "unknown argument(s): ['admin']"}),
 ('delete_all_orders', {'error': "unknown tool 'delete_all_orders'"}),
 ('get_order_status',
  {'error': 'arguments are not valid JSON: Expecting value: line 1 column 14 (char 13)'})]

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<h2 style="font-size:1.2em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">4. The protocol step by step</h2>
<p style="margin:.55em 0">Text in, so it's easy to follow.</p>

</div>

In [5]:
INSTRUCTIONS = "You are a support agent. Use tools for orders and temperatures. Be brief."
ws = await realtime.start_session(INSTRUCTIONS, tools=SPECS)
await realtime.send(ws, realtime.user_text("What's the status of order A123?"))
await realtime.send(ws, {"type": "response.create"})
ev = await realtime.collect(ws, until="response.done")
Counter(e['type'] for e in ev)

Counter({'response.function_call_arguments.delta': 12,
         'conversation.item.added': 2,
         'conversation.item.done': 2,
         'response.created': 1,
         'response.output_item.added': 1,
         'response.function_call_arguments.done': 1,
         'response.output_item.done': 1,
         'response.done': 1})

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<p style="margin:.55em 0">No audio deltas. The model's whole output this time is a <b>function call</b>, streamed as
<code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">function_call_arguments.delta</code> events:</p>

</div>

In [6]:
call = ev[-1]['response']['output'][0]
call['type'], call['name'], call['call_id'], call['arguments']

('function_call',
 'get_order_status',
 'call_8aLJS8tchPZ5rP37',
 '{  \n  "order_id": "A123"\n}')

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<p style="margin:.55em 0"><b>Our</b> code executes it:</p>

</div>

In [7]:
out = tools.run(REGISTRY, call['name'], call['arguments'])
out

{'result': {'status': 'shipped', 'arrives': 'Thursday'}}

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<p style="margin:.55em 0">Hand the result back, tied to the call by <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">call_id</code>, and ask the model to continue:</p>

</div>

In [8]:
result_item = {"type": "conversation.item.create",
               "item": {"type": "function_call_output", "call_id": call['call_id'], "output": json.dumps(out)}}
await realtime.send(ws, result_item)
await realtime.send(ws, {"type": "response.create"})
ev2 = await realtime.collect(ws, until="response.done")
[e['transcript'] for e in ev2 if e['type'] == 'response.output_audio_transcript.done']

['Your order A123 has been shipped and is expected to arrive on Thursday. If you need more details, let me know.']

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<h2 style="font-size:1.2em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">5. When the tool fails</h2>
<p style="margin:.55em 0"><code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">answer_tool_calls</code> wraps the last three cells. The error goes back as data, and the model explains it:</p>

</div>

In [9]:
await realtime.send(ws, realtime.user_text("And order Z999?"))
await realtime.send(ws, {"type": "response.create"})
ev = await realtime.collect(ws, until="response.done")
handled = await realtime.answer_tool_calls(ws, ev[-1]['response'], REGISTRY)
handled

ERROR:root:tool get_order_status failed
Traceback (most recent call last):
  File "D:\Code\realtime-talk\live\tools.py", line 54, in run
    return {"result": fn(**args)}
                      ^^^^^^^^^^
  File "C:\Users\aksha\AppData\Local\Temp\ipykernel_26548\2578210558.py", line 7, in get_order_status
    raise LookupError(f"no order {key}")
LookupError: no order Z999


[('get_order_status',
  '{  \n  "order_id": "Z999"\n}',
  {'error': 'LookupError: no order Z999'})]

In [10]:
ev2 = await realtime.collect(ws, until="response.done")
await ws.close()
[e['transcript'] for e in ev2 if e['type'] == 'response.output_audio_transcript.done']

["It looks like there's no order found with the ID Z999. Could you double-check the order ID or provide more details?"]

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<h2 style="font-size:1.2em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">6. Now by voice</h2>
<p style="margin:.55em 0">The same loop as notebook 04, with <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">tools=REGISTRY</code>. A simulated user reads the ID out loud, digit by digit:</p>

</div>

In [11]:
q = audio.say("Hi, can you check order A one two three for me?")
round(len(q) / audio.RATE, 2)

3.76

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<p style="margin:.55em 0">First <b>without</b> a filler (<code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">filler=False</code>), so we can measure the problem:</p>

</div>

In [12]:
spk = audio.Speaker(); spk.stream.start()
ws = await realtime.start_session(INSTRUCTIONS, tools=SPECS)
log_off = await realtime.converse(ws, audio.file_chunks(q), spk, seconds=len(q) / audio.RATE + 10, tools=REGISTRY, filler=False)
await ws.close()

0.42 input_audio_buffer.speech_started 0.00s of audio still queued


4.13 input_audio_buffer.speech_stopped 3584
4.13 input_audio_buffer.committed 
4.14 response.created in_progress


4.69 response.done completed
4.69 tool_call get_order_status({ "order_id": "A123" }) -> {"result": {"status": "shipped", "arrives": "Thursday"}}
4.69 user_said Hi, can you check order A123 for me?


5.06 response.created in_progress


5.53 first_audio_delta 


6.28 model_said Your order A123 has been shipped and is expected to arrive on Thursday. If you need more details, just let me know.
6.3 response.done completed


<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<h2 style="font-size:1.2em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">7. What a tool call costs in latency</h2>
<p style="margin:.55em 0">The model must finish one response (the call), we run the tool, then a <b>second</b> response speaks.
During that time the user hears silence:</p>

</div>

In [13]:
def first_sound(log):
    # seconds from end of user speech to the first thing the user hears (filler or model)
    t = {}
    for ts, ev_, _ in log:
        t.setdefault(ev_, ts)          # first occurrence of each event
    heard = min(t[k] for k in ('filler', 'first_audio_delta') if k in t)
    return round(heard - t['input_audio_buffer.speech_stopped'], 2)

silence_off = first_sound(log_off)
silence_off

1.4

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<p style="margin:.55em 0">Compare: about 0.45 s without tools (notebook 03). With a slow tool (a database or a web call),
this silence grows to several seconds, and the user wonders whether anything is happening.</p>
<h2 style="font-size:1.2em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">8. Filling the silence: "Let me check."</h2>
<p style="margin:.55em 0"><b>Deterministic and local</b>, not prompt-instructed: the moment the model <i>starts</i> streaming a function call
(<code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">response.output_item.added</code> with <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">item.type == "function_call"</code>), we push a pre-recorded clip
into the speaker. No extra model turn, no added latency, the same words every time.</p>
<p style="margin:.55em 0">The clip is generated once <b>in the model's own voice</b> (<code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">marin</code>), checked against its transcript,
and cached in <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">assets/</code>:</p>

</div>

In [14]:
filler = await realtime.load_filler()
filler.shape, round(len(filler) / audio.RATE, 2)

((32400,), 1.35)

In [15]:
Audio(filler, rate=audio.RATE)

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<p style="margin:.55em 0">Same question, with <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">filler=True</code> (the default whenever <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">tools</code> is given):</p>

</div>

In [16]:
ws = await realtime.start_session(INSTRUCTIONS, tools=SPECS)
log_on = await realtime.converse(ws, audio.file_chunks(q), spk, seconds=len(q) / audio.RATE + 10, tools=REGISTRY)
await ws.close()

0.41 input_audio_buffer.speech_started 0.00s of audio still queued


4.1 input_audio_buffer.speech_stopped 3584
4.1 input_audio_buffer.committed 
4.11 response.created in_progress


4.35 filler "Let me check." while get_order_status runs
4.42 response.done completed
4.42 tool_call get_order_status({ "order_id": "A123" }) -> {"result": {"status": "shipped", "arrives": "Thursday"}}


4.62 user_said Hi, can you check order A123 for me?
4.77 response.created in_progress


5.11 first_audio_delta 


5.69 model_said Your order A123 has been shipped and is expected to arrive on Thursday. If you need any more details, let me know!
5.69 response.done completed


In [17]:
[row for row in log_on if row[1] in ('input_audio_buffer.speech_stopped', 'filler', 'tool_call', 'first_audio_delta')]

[(4.1, 'input_audio_buffer.speech_stopped', 3584),
 (4.35, 'filler', '"Let me check." while get_order_status runs'),
 (4.42,
  'tool_call',
  'get_order_status({ "order_id": "A123" }) -> {"result": {"status": "shipped", "arrives": "Thursday"}}'),
 (5.11, 'first_audio_delta', '')]

In [18]:
silence_on = first_sound(log_on)
{'filler off': silence_off, 'filler on': silence_on}

{'filler off': 1.4, 'filler on': 0.25}

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<p style="margin:.55em 0">The answer queues <b>behind</b> the filler in the speaker buffer, so the two never overlap.</p>
<h3 style="font-size:1.05em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">Barge-in with a filler in the queue</h3>
<p style="margin:.55em 0">The queue now mixes filler audio with model audio, so "bytes written − bytes dropped" no longer says how much
of the <i>model's</i> message you heard. The Speaker therefore counts every byte it has played (<code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">spk.played</code>),
and each model message remembers where in that stream it started:</p>

</div>

In [19]:
spk.played, spk.position   # bytes played so far, where the next write lands

(640800, 640800)

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">


$$\mathrm{heard} = \operatorname{clamp}(\mathrm{played} - \mathrm{message\_start},\ 0,\ \mathrm{message\_length})$$

<ul style="margin:.4em 0;padding-left:1.4em">
  <li style="margin:.2em 0">Cut in during the filler: heard = 0 of the answer.</li>
  <li style="margin:.2em 0">Cut in after a message finished: no truncate is sent.</li>
</ul>

</div>

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<h2 style="font-size:1.2em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">9. Why this is deterministic, and where it isn't</h2>
<table style="border-collapse:collapse;margin:.8em 0">
  <tr><th style="border:1px solid rgba(127,127,127,.35);padding:4px 10px;text-align:left;background:rgba(127,127,127,.05)">Probabilistic (model)</th><th style="border:1px solid rgba(127,127,127,.35);padding:4px 10px;text-align:left;background:rgba(127,127,127,.05)">Deterministic (our code)</th></tr>
  <tr><td style="border:1px solid rgba(127,127,127,.35);padding:4px 10px;text-align:left;background:transparent"><i>whether</i> to call a tool, <i>which</i> one, the argument values</td><td style="border:1px solid rgba(127,127,127,.35);padding:4px 10px;text-align:left;background:transparent">which tools exist (registry = allowlist)</td></tr>
  <tr><td style="border:1px solid rgba(127,127,127,.35);padding:4px 10px;text-align:left;background:transparent">how it phrases the result</td><td style="border:1px solid rgba(127,127,127,.35);padding:4px 10px;text-align:left;background:transparent">argument validation, normalization, execution, error shape</td></tr>
  <tr><td style="border:1px solid rgba(127,127,127,.35);padding:4px 10px;text-align:left;background:transparent"></td><td style="border:1px solid rgba(127,127,127,.35);padding:4px 10px;text-align:left;background:transparent">barge-in: a cancelled response's calls are <b>never executed</b> (<code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">answer_tool_calls</code> skips them)</td></tr>
</table>
<div style="margin:.8em 0;padding:.4em 1em;border-left:4px solid #6366f1;background:rgba(99,102,241,.12);border-radius:0 6px 6px 0">For tools with side effects (payments, deletes), add a confirmation turn: the tool returns "needs confirmation",
and only a second call with <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">confirmed: true</code> acts.</div>

</div>

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<h2 style="font-size:1.2em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">10. Your turn 🎤🎧</h2>
<p style="margin:.55em 0">Try:</p>
<ul style="margin:.4em 0;padding-left:1.4em">
  <li style="margin:.2em 0">"What's the status of order B four five six?"</li>
  <li style="margin:.2em 0">"Convert 30 Celsius to Fahrenheit"</li>
  <li style="margin:.2em 0">an order that doesn't exist</li>
</ul>

</div>

In [20]:
LIVE_SECONDS = 45

In [21]:
ws = await realtime.start_session(INSTRUCTIONS, tools=SPECS)
live_log = await realtime.converse(ws, audio.mic_chunks(), spk, seconds=LIVE_SECONDS, tools=REGISTRY)
await ws.close()

In [22]:
live_log

[]

In [23]:
spk.stream.close()

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<span style="display:inline-block;margin-bottom:10px;padding:2px 12px;border-radius:999px;background:#6366f1;color:#fff;font:600 11px/20px system-ui,sans-serif;letter-spacing:.05em;text-transform:uppercase">Takeaways</span>
<ul style="margin:.4em 0;padding-left:1.4em">
  <li style="margin:.2em 0">Tool calling on a live socket = <b><code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">response.done</code> carries function_call items → we send
      <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">function_call_output</code> → <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">response.create</code></b>.</li>
  <li style="margin:.2em 0">The model proposes and our code disposes: registry, validation, and error-as-data make execution deterministic.</li>
  <li style="margin:.2em 0">Every tool call adds a second model turn. A local, pre-recorded filler fills that silence deterministically.</li>
</ul>
<p style="margin:.55em 0"><b>Next (06):</b> memory. Facts that survive across sessions, built from the same tool mechanism
(<code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">remember</code> / <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">recall</code>) plus instructions injected at session start.</p>

</div>